# 02 · Análise exploratória dos CVLI em Alagoas (2013–2025)

Este notebook responde às perguntas de pesquisa do README. Ele parte da mesma limpeza do `01_preprocessamento.ipynb` (recorte até 2025, idade `NI` → `NaN`, faixas etárias) e acrescenta o que falta: padronização de local, horário, município e feminicídio.

**Limitações a ter em mente ao ler os resultados**

- A base não traz população. Todas as comparações entre municípios usam **números absolutos** e variação percentual, não taxa por 100 mil habitantes.
- A classificação *Feminicídio* só aparece a partir de 2015 (Lei 13.104/2015). Antes disso, esses casos entram como *Homicídio*.
- `LOCAL DO FATO` é texto livre, com centenas de grafias. Foi preciso agrupar.

In [ ]:
import re
import unicodedata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({
    "figure.dpi": 110,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.axisbelow": True,
})
COR = "#1f4e79"      # cor principal
COR2 = "#c0504d"     # destaque
CINZA = "#8a8a8a"

## 1. Carga e limpeza

In [ ]:
df = pd.read_csv("../dados_brutos/cvli_dados.csv")
df["DATA DO FATO"] = pd.to_datetime(df["DATA DO FATO"], dayfirst=True)
df = df[df["DATA DO FATO"] <= "2025-12-31"].copy()   # 2026 está incompleto

df["ANO"] = df["DATA DO FATO"].dt.year
df["IDADE_NUMERICA"] = pd.to_numeric(df["IDADE DA VITIMA"], errors="coerce")
df["HORA"] = pd.to_numeric(df["HORA DO FATO"].str[:2], errors="coerce")

print(df.shape)
print("IDs duplicados:", df["ID_CONTROLE"].duplicated().sum())

### Casos por ano

2012 fica fora das comparações porque a série só passa a ser regular a partir de 2013 (decisão já tomada no notebook 01).

In [ ]:
por_ano = df.groupby("ANO").size()
display(por_ano.to_frame("casos").T)

fig, ax = plt.subplots(figsize=(10, 4))
por_ano.loc[2013:].plot(kind="bar", color=COR, ax=ax, width=0.75)
for i, v in enumerate(por_ano.loc[2013:]):
    ax.text(i, v + 20, f"{v:,}".replace(",", "."), ha="center", fontsize=8)
ax.set_title("CVLI por ano em Alagoas, 2013–2025")
ax.set_xlabel("")
ax.set_ylabel("Vítimas")
ax.tick_params(axis="x", rotation=0)
plt.tight_layout()
plt.show()

q = (por_ano[2025] / por_ano[2013] - 1) * 100
print(f"Variação 2013 → 2025: {q:.1f}%")

### Qualidade dos dados: valores `NI` e outros marcadores de ausência

In [ ]:
marcadores = {"NI", "N/I", "Sem Informação", "Não Informado", "Não informado"}
ausentes = pd.DataFrame({
    "NI": [df[c].isin(marcadores).sum() for c in
           ["COR/RACA DA VITIMA", "IDADE DA VITIMA", "INSTRUMENTO UTILIZADO",
            "LOCAL DO FATO", "OCUPACAO VITIMA", "ESCOLARIDADE VITIMA"]]
}, index=["Cor/raça", "Idade", "Instrumento", "Local", "Ocupação", "Escolaridade"])
ausentes["% da base"] = (ausentes["NI"] / len(df) * 100).round(1)
display(ausentes)

Escolaridade e ocupação têm ausência muito alta e **não serão usadas** para tirar conclusões. Cor/raça, idade, instrumento e local têm cobertura boa.

## 2. Perfil das vítimas

In [ ]:
sexo = df["SEXO DA VITIMA"].value_counts()
raca = df["COR/RACA DA VITIMA"].value_counts()
instr = df["INSTRUMENTO UTILIZADO"].value_counts()

fig, axs = plt.subplots(1, 3, figsize=(14, 4))
for ax, serie, titulo in zip(axs, [sexo, raca, instr.head(6)], ["Sexo", "Cor/raça", "Instrumento (6 mais comuns)"]):
    pct = serie / serie.sum() * 100
    pct.sort_values().plot(kind="barh", ax=ax, color=COR)
    for i, v in enumerate(pct.sort_values()):
        ax.text(v + 0.5, i, f"{v:.1f}%", va="center", fontsize=8)
    ax.set_title(titulo)
    ax.set_xlim(0, 100)
    ax.set_ylabel("")
plt.tight_layout()
plt.show()

PAF = projétil de arma de fogo; BRANCA = arma branca. Abaixo, como o instrumento mudou ao longo do tempo.

In [ ]:
instr_ano = pd.crosstab(
    df.loc[df["ANO"] >= 2013, "ANO"],
    df.loc[df["ANO"] >= 2013, "INSTRUMENTO UTILIZADO"].where(lambda s: s.isin(["PAF", "BRANCA", "ESPANCAMENTO"]), "Outros/NI"),
    normalize="index",
) * 100
instr_ano.plot(kind="line", marker="o", figsize=(10, 4))
plt.title("Participação por instrumento, % das vítimas")
plt.ylabel("%")
plt.xlabel("")
plt.legend(title="")
plt.tight_layout()
plt.show()

## 3. Pergunta 1 · A queda foi homogênea entre os municípios?

Compara-se a **média anual de 2013–2015** com a de **2023–2025** (três anos em cada ponta, para suavizar oscilações de municípios pequenos). Foram considerados os municípios com pelo menos 150 vítimas no período.

In [ ]:
d = df[df["ANO"].between(2013, 2025)]
ini = d[d["ANO"].between(2013, 2015)].groupby("CIDADE DO FATO").size() / 3
fim = d[d["ANO"].between(2023, 2025)].groupby("CIDADE DO FATO").size() / 3
tot = d.groupby("CIDADE DO FATO").size()

mun = pd.DataFrame({"total": tot, "media_2013_15": ini, "media_2023_25": fim}).fillna(0)
mun["variacao_%"] = (mun["media_2023_25"] / mun["media_2013_15"] - 1) * 100
mun = mun.replace([np.inf, -np.inf], np.nan)

grandes = mun[mun["total"] >= 150].sort_values("variacao_%")
display(grandes.round(1))
print(f"{len(grandes)} municípios com ≥150 vítimas; "
      f"{(grandes['variacao_%'] < 0).sum()} tiveram queda, {(grandes['variacao_%'] >= 0).sum()} subiram ou ficaram estáveis")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
cores = [COR if v < 0 else COR2 for v in grandes["variacao_%"]]
ax.barh(grandes.index, grandes["variacao_%"], color=cores)
ax.axvline(0, color="black", lw=0.8)
for i, v in enumerate(grandes["variacao_%"]):
    ax.text(v + (2 if v >= 0 else -2), i, f"{v:.0f}%", va="center", ha="left" if v >= 0 else "right", fontsize=8)
ax.set_title("Variação da média anual de vítimas: 2013–15 → 2023–25")
ax.set_xlabel("%")
ax.grid(axis="y", visible=False)
plt.tight_layout()
plt.show()

### Maceió, Arapiraca e interior ao longo do tempo

In [ ]:
grupo = d["CIDADE DO FATO"].where(d["CIDADE DO FATO"].isin(["Maceió", "Arapiraca"]), "Demais municípios")
g = pd.crosstab(d["ANO"], grupo)
g.plot(kind="line", marker="o", figsize=(10, 4))
plt.title("Vítimas por ano: capital, Arapiraca e demais municípios")
plt.ylabel("Vítimas")
plt.xlabel("")
plt.legend(title="")
plt.tight_layout()
plt.show()

indice = g / g.loc[2013] * 100
display(indice.round(0).astype(int).T)

## 4. Pergunta 5 e 6 · Concentração territorial

Sem população não dá para falar em taxa. O que a base permite é medir **quanto da violência se concentra em poucos municípios** (e bairros) e se isso mudou.

In [ ]:
acum = (tot.sort_values(ascending=False).cumsum() / tot.sum() * 100)
print(f"Municípios com registro: {tot.shape[0]} de 102 em Alagoas")
for n in (1, 3, 5, 10, 20):
    print(f"Top {n:>2} municípios concentram {acum.iloc[n-1]:.1f}% das vítimas")

top = tot.sort_values(ascending=False).head(15)
fig, ax = plt.subplots(figsize=(9, 5))
(top.sort_values() / tot.sum() * 100).plot(kind="barh", color=COR, ax=ax)
ax.set_title("Participação no total de vítimas, 2013–2025 (15 maiores)")
ax.set_xlabel("%")
ax.set_ylabel("")
plt.tight_layout()
plt.show()

In [ ]:
share_mac = (d["CIDADE DO FATO"].eq("Maceió").groupby(d["ANO"]).mean() * 100)
top5 = tot.sort_values(ascending=False).head(5).index
share_top5 = (d["CIDADE DO FATO"].isin(top5).groupby(d["ANO"]).mean() * 100)

fig, ax = plt.subplots(figsize=(10, 4))
share_mac.plot(marker="o", color=COR, label="Maceió", ax=ax)
share_top5.plot(marker="o", color=COR2, label="5 maiores municípios", ax=ax)
ax.set_title("Parcela dos casos concentrada na capital e nos 5 maiores municípios")
ax.set_ylabel("% das vítimas do ano")
ax.set_xlabel("")
ax.legend()
plt.tight_layout()
plt.show()
print("Top 5:", list(top5))

### Bairros (Maceió)

`Zona Rural` e `Centro` aparecem como bairro em vários municípios; por isso a análise por bairro é feita só em Maceió.

In [ ]:
mac = d[d["CIDADE DO FATO"] == "Maceió"]
bairros = mac["BAIRRO DO FATO"].value_counts()
print(f"{bairros.shape[0]} bairros distintos em Maceió; top 10 = {bairros.head(10).sum()/len(mac)*100:.1f}% dos casos")

fig, ax = plt.subplots(figsize=(9, 5))
(bairros.head(15).sort_values()).plot(kind="barh", color=COR, ax=ax)
ax.set_title("Maceió: bairros com mais vítimas, 2013–2025")
ax.set_xlabel("Vítimas")
ax.set_ylabel("")
plt.tight_layout()
plt.show()

## 5. Pergunta 2 · Feminicídios acompanharam a queda?

Duas medidas, porque a classificação *Feminicídio* só existe a partir de 2015:

1. **Feminicídios classificados** (2015 em diante).
2. **Todas as vítimas mulheres** (série completa), que não depende da classificação.

In [ ]:
d["FEM"] = d["SUBJETIVIDADE COMPLEMENTAR"].eq("Feminicídio")
d["MULHER"] = d["SEXO DA VITIMA"].eq("Feminino")

fem = d.groupby("ANO").agg(total=("ID_CONTROLE", "size"),
                           mulheres=("MULHER", "sum"),
                           feminicidios=("FEM", "sum"))
fem["%_mulheres"] = fem["mulheres"] / fem["total"] * 100
fem["feminicidio_/_mulheres_%"] = fem["feminicidios"] / fem["mulheres"] * 100
display(fem.round(1))

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(14, 4.2))

base = fem[["total", "mulheres"]] / fem.loc[2013, ["total", "mulheres"]] * 100
axs[0].plot(base.index, base["total"], marker="o", color=COR, label="Todas as vítimas")
axs[0].plot(base.index, base["mulheres"], marker="o", color=COR2, label="Vítimas mulheres")
axs[0].axhline(100, color=CINZA, lw=0.8)
axs[0].set_title("Índice (2013 = 100)")
axs[0].legend()

axs[1].bar(fem.index, fem["mulheres"], color=CINZA, label="Mulheres vítimas")
axs[1].bar(fem.index, fem["feminicidios"], color=COR2, label="Classificadas como feminicídio")
axs[1].set_title("Mulheres vítimas e feminicídios classificados")
axs[1].legend()
for ax in axs:
    ax.set_xticks(fem.index)
    ax.tick_params(axis="x", rotation=60)
plt.tight_layout()
plt.show()

In [ ]:
print(f"Total de vítimas: {fem.loc[2013,'total']} → {fem.loc[2025,'total']} ({(fem.loc[2025,'total']/fem.loc[2013,'total']-1)*100:.0f}%)")
print(f"Mulheres:         {fem.loc[2013,'mulheres']} → {fem.loc[2025,'mulheres']} ({(fem.loc[2025,'mulheres']/fem.loc[2013,'mulheres']-1)*100:.0f}%)")
print(f"Mulheres como % do total: {fem.loc[2013,'%_mulheres']:.1f}% (2013) → {fem.loc[2025,'%_mulheres']:.1f}% (2025)")

## 6. Pergunta 3 · Horário dos crimes

Períodos: madrugada (0h–5h), manhã (6h–11h), tarde (12h–17h) e noite (18h–23h).

Os registros em  foram checados contra os minutos cheios vizinhos (23:00 e 01:00) para ver se havia horário padrão preenchido por falta de informação; a checagem está na saída da célula abaixo.

In [ ]:
h = d.dropna(subset=["HORA"]).copy()
h["HORA"] = h["HORA"].astype(int)
por_hora = h["HORA"].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(10, 4))
cores = [COR2 if 18 <= x <= 23 else COR for x in por_hora.index]
ax.bar(por_hora.index, por_hora.values, color=cores)
ax.set_xticks(range(24))
ax.set_title("Vítimas por hora do dia, 2013–2025 (noite em destaque)")
ax.set_xlabel("Hora")
ax.set_ylabel("Vítimas")
plt.tight_layout()
plt.show()

bins = [-1, 5, 11, 17, 23]
h["PERIODO"] = pd.cut(h["HORA"], bins=bins, labels=["Madrugada", "Manhã", "Tarde", "Noite"])
per = h["PERIODO"].value_counts(normalize=True).reindex(["Madrugada", "Manhã", "Tarde", "Noite"]) * 100
display(per.round(1).to_frame("% das vítimas"))

zero = (h["HORA DO FATO"] == "00:00").sum()
print(f"Registros em 00:00: {zero} | 23:00: {(h['HORA DO FATO']=='23:00').sum()} | 01:00: {(h['HORA DO FATO']=='01:00').sum()}  (sem excesso aparente em 00:00)")

In [ ]:
per_ano = pd.crosstab(h["ANO"], h["PERIODO"], normalize="index") * 100
per_ano.plot(kind="line", marker="o", figsize=(10, 4))
plt.title("Período do dia ao longo dos anos (% das vítimas)")
plt.ylabel("%")
plt.xlabel("")
plt.legend(title="")
plt.tight_layout()
plt.show()

print("Período por sexo da vítima (%):")
display((pd.crosstab(h["SEXO DA VITIMA"], h["PERIODO"], normalize="index") * 100).round(1))

## 7. Pergunta 4 · Local público ou privado?

`LOCAL DO FATO` tem centenas de grafias (inclusive links de notícias e erros de digitação). Elas foram normalizadas e agrupadas em categorias. Como "imediações" e "porta de casa" ficam entre o público e o privado, foram mantidas como categoria separada.

In [ ]:
def sem_acento(s):
    return unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode().lower().strip()

def classifica_local(x):
    s = sem_acento(str(x))
    if s in {"ni", "n/i", "sem informacao", "nao informado", "outro", "outros", ""} or s.startswith("http"):
        return "Não informado/outros"
    if re.search(r"show|evento|aposta|hotel|motel|hospedagem|comercial|coercial|ciomercial|bar\b|restaurante|mercado|oficina|"
                 r"salao|barbearia|boate|danceteria|igreja|escola|ensino|saude|hospital|hge|consultorio|clinica|"
                 r"trabalho|feira|clube|quadra|campo de futebol|churrasquinho|estabelecimento|shopping|"
                 r"presidio|cela|cadeia|cadeiao|delegacia|detencao|custodia|internacao|prisional|orgao|instituicao|centro social|"
                 r"parque|mirante|bloco|aviacao|apoio|assistencia|neas|rodoviaria", s):
        return "Estabelecimento/instituição"
    if re.search(r"imedia|proxi|porta de|exterior|quintal|arredores|esquina|trás do|tras do", s):
        return "Entorno da residência"
    if re.search(r"casa|resid|resod|privad|quarto|domicilio|telhado", s) and not re.search(r"abandon|sem morador|construcao", s):
        return "Dentro da residência"
    if re.search(r"^v[ií]a|^via|logradouro|local p|loca p|publi|public|praca|pracinha|beco|travessa|rodovia|estrada|ponto de onibus|"
                 r"estacionamento|coletivo|onibus|van\b|alternativo|taxi|veiculo|ferrovia|linha ferrea|calcad|vila$|canal|rua", s):
        return "Via/espaço público"
    if re.search(r"vegeta|vegta|mata|plantac|canavial|zona rural|area rural|rural|sitio|fazenda|chacara|engenho|propriedade|terras|"
                 r"rio\b|riacho|lagoa|acude|barragem|cacimba|corrego|grota|mangue|praia|margem|beira|vala|valeta|fossa|cova|"
                 r"terreno|tereno|baldio|area verde|ribanceira|buraco|lixao|aquatico|campo", s):
        return "Área rural/natural/terreno"
    if re.search(r"abandon|desocupad|sem morador|construcao|obra|galpao|edificacao|predio|alvenaria|imovel", s):
        return "Imóvel abandonado/em obra"
    return "Não informado/outros"

df["LOCAL_GRUPO"] = df["LOCAL DO FATO"].map(classifica_local)
d = df[df["ANO"].between(2013, 2025)].copy()   # recria com a nova coluna
d["FEM"] = d["SUBJETIVIDADE COMPLEMENTAR"].eq("Feminicídio")
d["MULHER"] = d["SEXO DA VITIMA"].eq("Feminino")

loc = d["LOCAL_GRUPO"].value_counts()
display((loc / loc.sum() * 100).round(1).to_frame("%"))

print("Maiores grafias que caíram em 'Não informado/outros':")
display(d.loc[d["LOCAL_GRUPO"] == "Não informado/outros", "LOCAL DO FATO"].value_counts().head(10))

In [ ]:
ordem = ["Via/espaço público", "Dentro da residência", "Entorno da residência",
         "Área rural/natural/terreno", "Estabelecimento/instituição",
         "Imóvel abandonado/em obra", "Não informado/outros"]
fig, ax = plt.subplots(figsize=(9, 4.2))
(loc.reindex(ordem)[::-1] / loc.sum() * 100).plot(kind="barh", color=COR, ax=ax)
ax.set_title("Onde ocorreram os casos, 2013–2025 (%)")
ax.set_xlabel("%")
ax.set_ylabel("")
plt.tight_layout()
plt.show()

print("Local por sexo da vítima (%):")
display((pd.crosstab(d["SEXO DA VITIMA"], d["LOCAL_GRUPO"], normalize="index") * 100).round(1)[ordem])

print("Local entre feminicídios classificados (%):")
display((d.loc[d["FEM"], "LOCAL_GRUPO"].value_counts(normalize=True) * 100).round(1).to_frame("%"))

In [ ]:
local_ano = pd.crosstab(d["ANO"], d["LOCAL_GRUPO"], normalize="index") * 100
local_ano[["Via/espaço público", "Dentro da residência", "Entorno da residência", "Área rural/natural/terreno"]].plot(
    kind="line", marker="o", figsize=(10, 4))
plt.title("Local do fato ao longo dos anos (% das vítimas)")
plt.ylabel("%")
plt.xlabel("")
plt.legend(title="", fontsize=8)
plt.tight_layout()
plt.show()

# Etapa II · Tratamento e análise descritiva

## 8. Qualidade dos dados e tipagem

In [ ]:
# Período do dia no dataframe principal de análise (2013–2025)
d["PERIODO"] = pd.cut(d["HORA"], bins=[-1, 5, 11, 17, 23], labels=["Madrugada", "Manhã", "Tarde", "Noite"], ordered=True)

# NI -> NaN nas colunas categóricas (o 'NI' é ausência, não uma categoria)
NI = {"NI", "N/I", "Sem Informação", "Não Informado", "Não informado"}
for c in ["COR/RACA DA VITIMA", "INSTRUMENTO UTILIZADO", "ESCOLARIDADE VITIMA", "OCUPACAO VITIMA"]:
    d[c] = d[c].where(~d[c].isin(NI))

# Tipagem
d["FAIXA_ETARIA"] = pd.cut(d["IDADE_NUMERICA"], [-1, 11, 17, 24, 34, 44, 59, np.inf],
                           labels=["0–11", "12–17", "18–24", "25–34", "35–44", "45–59", "60+"], ordered=True)
d["ESCOLARIDADE VITIMA"] = pd.Categorical(d["ESCOLARIDADE VITIMA"],
                                          categories=["Analfabeto", "Alfabetizado", "Fundamental", "Médio", "Superior"], ordered=True)
for c in ["SEXO DA VITIMA", "COR/RACA DA VITIMA", "INSTRUMENTO UTILIZADO", "CIDADE DO FATO", "LOCAL_GRUPO",
          "SUBJETIVIDADE COMPLEMENTAR"]:
    d[c] = d[c].astype("category")

tipos = pd.DataFrame([
    ("IDADE_NUMERICA", "Quantitativa discreta", "anos completos"),
    ("HORA", "Quantitativa discreta (circular)", "0–23"),
    ("ANO", "Quantitativa discreta / temporal", "2013–2025"),
    ("DATA DO FATO", "Temporal", "data"),
    ("SEXO DA VITIMA", "Qualitativa nominal", "2 categorias"),
    ("COR/RACA DA VITIMA", "Qualitativa nominal", "categorias IBGE"),
    ("CIDADE DO FATO / BAIRRO DO FATO", "Qualitativa nominal", "102 municípios"),
    ("INSTRUMENTO UTILIZADO", "Qualitativa nominal", ""),
    ("SUBJETIVIDADE COMPLEMENTAR", "Qualitativa nominal", "tipo penal"),
    ("LOCAL_GRUPO", "Qualitativa nominal", "agrupado a partir de texto livre"),
    ("FAIXA_ETARIA", "Qualitativa ordinal", "7 faixas"),
    ("ESCOLARIDADE VITIMA", "Qualitativa ordinal", "analfabeto < ... < superior"),
    ("PERIODO", "Qualitativa ordinal", "madrugada < manhã < tarde < noite"),
], columns=["variável", "tipo", "observação"])
display(tipos)

### Valores ausentes, duplicatas e valores inválidos

In [ ]:
aus = pd.DataFrame({"ausentes": d.isna().sum(), "%": (d.isna().mean() * 100).round(1)})
display(aus[aus["ausentes"] > 0].sort_values("ausentes", ascending=False))

print("Duplicatas por ID_CONTROLE:", d["ID_CONTROLE"].duplicated().sum())
print("Linhas idênticas em todas as colunas exceto o ID:", d.duplicated(subset=[c for c in d.columns if c in df.columns and c != "ID_CONTROLE"]).sum())
print("Horas fora de 0–23:", (~d["HORA"].dropna().between(0, 23)).sum(), "| datas no futuro:", (d["DATA DO FATO"] > pd.Timestamp.today()).sum())

**Tratamento adotado**

- Ausentes: `NI` virou `NaN`. Idade ausente (2%) fica fora das médias; escolaridade e ocupação, com mais de 65% ausente, não entram nas inferências.
- Duplicatas por ID: nenhuma. Linhas idênticas em todas as colunas exceto o ID podem ser vítimas diferentes do mesmo evento (por exemplo, chacina), então **não foram removidas**.

### Outliers na idade

In [ ]:
idade = d["IDADE_NUMERICA"].dropna()
q1, q3 = idade.quantile([0.25, 0.75])
iqr = q3 - q1
lim_sup = q3 + 1.5 * iqr
out = idade[idade > lim_sup]
print(f"Q1={q1:.0f}, Q3={q3:.0f}, IQR={iqr:.0f}; limite superior de Tukey = {lim_sup:.0f} anos")
print(f"Outliers superiores: {len(out)} ({len(out)/len(idade)*100:.1f}%); idade máxima = {idade.max():.0f}")
print("Idades ≥ 95:", sorted(idade[idade >= 95].astype(int).tolist()))

Os valores acima do limite de Tukey são vítimas idosas reais, apenas raras; as idades máximas (acima de 95) são improváveis, mas possíveis. Como não há evidência de erro, os outliers **foram mantidos**, e a análise usa mediana e testes robustos em paralelo à média.

## 9. Estatística descritiva univariada

In [ ]:
def resumo(s):
    s = s.dropna()
    return pd.Series({
        "n": len(s), "média": s.mean(), "mediana": s.median(), "desvio padrão": s.std(),
        "Q1": s.quantile(.25), "Q3": s.quantile(.75), "IQR": s.quantile(.75) - s.quantile(.25),
        "assimetria": s.skew(), "mínimo": s.min(), "máximo": s.max(),
    })

tab_num = pd.DataFrame({
    "Idade (anos)": resumo(d["IDADE_NUMERICA"]),
    "Hora do fato (0–23)": resumo(d["HORA"]),
    "Vítimas por mês": resumo(d.groupby(d["DATA DO FATO"].dt.to_period("M")).size()),
}).T
display(tab_num.round(2))

A idade tem assimetria positiva (cauda de vítimas mais velhas): a média fica acima da mediana. A hora do dia é uma variável circular (23h e 0h são vizinhas), então média e desvio padrão dela servem só como referência; a distribuição por hora na seção 6 é mais informativa.

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(12, 4))
axs[0].hist(idade, bins=range(0, 105, 2), color=COR, edgecolor="white")
axs[0].axvline(idade.mean(), color=COR2, ls="--", label=f"média {idade.mean():.1f}")
axs[0].axvline(idade.median(), color="black", ls=":", label=f"mediana {idade.median():.0f}")
axs[0].set_title("Distribuição da idade das vítimas")
axs[0].set_xlabel("Idade (anos)")
axs[0].set_ylabel("Vítimas")
axs[0].legend()
axs[1].boxplot(idade, vert=False, widths=0.5, flierprops=dict(marker=".", markersize=3, alpha=0.4))
axs[1].set_title("Boxplot da idade")
axs[1].set_xlabel("Idade (anos)")
axs[1].set_yticks([])
plt.tight_layout()
plt.show()

In [ ]:
def freq(col):
    t = d[col].value_counts(dropna=False)
    out = pd.DataFrame({"frequência": t, "%": (t / t.sum() * 100).round(1)})
    out.index = out.index.astype(object).fillna("(ausente)")
    return out

for col in ["SEXO DA VITIMA", "SUBJETIVIDADE COMPLEMENTAR", "COR/RACA DA VITIMA", "INSTRUMENTO UTILIZADO",
            "FAIXA_ETARIA", "PERIODO", "LOCAL_GRUPO", "ESCOLARIDADE VITIMA"]:
    print(f"\n{col}")
    display(freq(col).head(10))

## 10. Análise bivariada

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(15, 4.2))

# numérica x categórica: idade por sexo
dados = [d.loc[d["SEXO DA VITIMA"] == s, "IDADE_NUMERICA"].dropna() for s in ["Masculino", "Feminino"]]
axs[0].boxplot(dados, tick_labels=["Masculino", "Feminino"], flierprops=dict(marker=".", markersize=3, alpha=0.3))
axs[0].set_title("Idade por sexo")
axs[0].set_ylabel("Idade (anos)")

# numérica x categórica: idade por instrumento
inst = ["PAF", "BRANCA", "ESPANCAMENTO"]
dados = [d.loc[d["INSTRUMENTO UTILIZADO"] == i, "IDADE_NUMERICA"].dropna() for i in inst]
axs[1].boxplot(dados, tick_labels=inst, flierprops=dict(marker=".", markersize=3, alpha=0.3))
axs[1].set_title("Idade por instrumento")

# categórica x categórica: período por sexo (barras empilhadas, %)
ct = pd.crosstab(d["SEXO DA VITIMA"], d["PERIODO"], normalize="index") * 100
ct.plot(kind="bar", stacked=True, ax=axs[2], colormap="Blues", edgecolor="white")
axs[2].set_title("Período do dia por sexo (%)")
axs[2].set_xlabel("")
axs[2].tick_params(axis="x", rotation=0)
axs[2].legend(title="", fontsize=8)
plt.tight_layout()
plt.show()

Para a **matriz de correlação** é preciso variáveis quantitativas. Em nível de vítima só há idade, hora e ano, o que é pouco. Por isso a unidade de análise passa a ser o **município** (os que têm pelo menos 100 vítimas em 2013–2025), com indicadores quantitativos calculados para cada um. Usa-se **Spearman**, porque a distribuição dos totais é muito assimétrica (Maceió domina) e o interesse é a associação monotônica.

In [ ]:
d["NOITE"] = (d["PERIODO"] == "Noite").astype(int)
d["PAF"] = (d["INSTRUMENTO UTILIZADO"] == "PAF").astype(int)
d["RESID"] = (d["LOCAL_GRUPO"] == "Dentro da residência").astype(int)
d["MULHER_I"] = (d["SEXO DA VITIMA"] == "Feminino").astype(int)

m = d.groupby("CIDADE DO FATO", observed=True).agg(
    vitimas=("ID_CONTROLE", "size"),
    idade_media=("IDADE_NUMERICA", "mean"),
    pct_mulheres=("MULHER_I", "mean"),
    pct_noite=("NOITE", "mean"),
    pct_arma_fogo=("PAF", "mean"),
    pct_dentro_casa=("RESID", "mean"),
)
m = m[m["vitimas"] >= 100]
for c in ["pct_mulheres", "pct_noite", "pct_arma_fogo", "pct_dentro_casa"]:
    m[c] *= 100
# variação 2013-15 -> 2023-25 por município
ini = d[d["ANO"].between(2013, 2015)].groupby("CIDADE DO FATO", observed=True).size()
fim = d[d["ANO"].between(2023, 2025)].groupby("CIDADE DO FATO", observed=True).size()
m["variacao_pct"] = ((fim / ini - 1) * 100).reindex(m.index)

print(f"{len(m)} municípios na análise")
corr = m.corr(method="spearman")
display(corr.round(2))

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr)), corr.columns, rotation=45, ha="right")
ax.set_yticks(range(len(corr)), corr.columns)
for i in range(len(corr)):
    for j in range(len(corr)):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=8)
ax.grid(False)
fig.colorbar(im, ax=ax, shrink=0.8)
ax.set_title("Correlação de Spearman entre indicadores municipais")
plt.tight_layout()
plt.show()

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(12, 4.2))
axs[0].scatter(m["vitimas"], m["pct_arma_fogo"], color=COR, alpha=0.7)
axs[0].set_xscale("log")
axs[0].set_xlabel("Vítimas 2013–2025 (escala log)")
axs[0].set_ylabel("% por arma de fogo")
axs[0].set_title("Volume de casos × uso de arma de fogo")
axs[1].scatter(m["vitimas"], m["variacao_pct"], color=COR, alpha=0.7)
axs[1].set_xscale("log")
axs[1].axhline(0, color=CINZA, lw=0.8)
axs[1].set_xlabel("Vítimas 2013–2025 (escala log)")
axs[1].set_ylabel("Variação 2013–15 → 2023–25 (%)")
axs[1].set_title("Volume de casos × variação")
plt.tight_layout()
plt.show()

Com 41 municípios, correlações abaixo de cerca de |0,3| não se distinguem do acaso. As mais fortes envolvem poucos pares (por exemplo, volume de casos × % de mulheres, ρ ≈ 0,31); nenhum indicador está fortemente associado à variação 2013–15 → 2023–25, ou seja, o tamanho do município não prevê o tamanho da queda. Correlação entre indicadores municipais não implica causalidade e não vale para indivíduos.

# Etapa III · Análise inferencial

Convenções: nível de significância **α = 0,05** em todos os testes. A base é um **censo** de vítimas registradas, não uma amostra aleatória; os intervalos e testes tratam o conjunto de casos como amostra de um processo gerador subjacente (o que seria observado em outros anos ou condições). Com n grande, diferenças minúsculas ficam "significativas", por isso cada teste vem com uma **medida de tamanho de efeito**.

## 11. Intervalos de confiança (95%)

In [ ]:
from scipy import stats

def ic_prop(k, n, nome):
    r = stats.binomtest(int(k), int(n)).proportion_ci(confidence_level=0.95, method="wilson")
    return {"indicador": nome, "n": int(n), "estimativa": k / n * 100, "IC95% inf": r.low * 100, "IC95% sup": r.high * 100}

def ic_media(s, nome):
    s = s.dropna()
    lo, hi = stats.t.interval(0.95, len(s) - 1, loc=s.mean(), scale=stats.sem(s))
    return {"indicador": nome, "n": len(s), "estimativa": s.mean(), "IC95% inf": lo, "IC95% sup": hi}

per = d["PERIODO"].notna()
linhas = [
    ic_prop(d["NOITE"][per].sum(), per.sum(), "% de casos à noite (18h–23h)"),
    ic_prop((d["LOCAL_GRUPO"] == "Via/espaço público").sum(), len(d), "% em via/espaço público"),
    ic_prop(d["RESID"].sum(), len(d), "% dentro da residência"),
    ic_prop(d["MULHER_I"].sum(), len(d), "% de vítimas mulheres (2013–2025)"),
    ic_prop(d.loc[d["ANO"] <= 2015, "MULHER_I"].sum(), (d["ANO"] <= 2015).sum(), "% mulheres, 2013–2015"),
    ic_prop(d.loc[d["ANO"] >= 2023, "MULHER_I"].sum(), (d["ANO"] >= 2023).sum(), "% mulheres, 2023–2025"),
    ic_prop(d["PAF"].sum(), d["INSTRUMENTO UTILIZADO"].notna().sum(), "% por arma de fogo"),
    ic_media(d["IDADE_NUMERICA"], "Idade média (anos), todas"),
    ic_media(d.loc[d["SEXO DA VITIMA"] == "Masculino", "IDADE_NUMERICA"], "Idade média, homens"),
    ic_media(d.loc[d["SEXO DA VITIMA"] == "Feminino", "IDADE_NUMERICA"], "Idade média, mulheres"),
]
ics = pd.DataFrame(linhas).set_index("indicador")
display(ics.round(2))

**Leitura.** Os intervalos são estreitos por causa do tamanho da base. O ponto a observar é se eles se sobrepõem: se o intervalo de `% mulheres, 2013–2015` não cobre o de `2023–2025`, isso é um indício (confirmado formalmente no Teste 5) de que a proporção de mulheres entre as vítimas mudou. Já o intervalo de `% de casos à noite` mostra se a noite concentra mais que os 25% esperados caso o horário fosse indiferente.

## 12. Testes de hipóteses

In [ ]:
ALPHA = 0.05

def decisao(p):
    return "Rejeita H0" if p < ALPHA else "Não rejeita H0"

def cramers_v(tab):
    chi2 = stats.chi2_contingency(tab, correction=False)[0]
    n = np.asarray(tab).sum()
    return np.sqrt(chi2 / (n * (min(np.asarray(tab).shape) - 1)))

resultados = []
def registra(teste, estat, p, efeito, pergunta):
    resultados.append({"teste": teste, "estatística": estat, "p-valor": p, "efeito": efeito, "decisão (α=0,05)": decisao(p), "pergunta": pergunta})

### Teste 1 · O horário do crime é indiferente? (Pergunta 3)

- **H0:** os casos se distribuem igualmente entre madrugada, manhã, tarde e noite (25% cada; os quatro períodos têm 6 horas).
- **H1:** a distribuição não é uniforme.
- **Teste:** qui-quadrado de aderência, porque a variável é categórica e a hipótese nula define proporções esperadas.

In [ ]:
obs = d["PERIODO"].value_counts().reindex(["Madrugada", "Manhã", "Tarde", "Noite"])
chi2, p = stats.chisquare(obs)
w = np.sqrt(chi2 / obs.sum() / (len(obs) - 1))
display(pd.DataFrame({"observado": obs, "esperado": obs.sum() / 4, "% observado": (obs / obs.sum() * 100).round(1)}))
print(f"χ²({len(obs)-1}) = {chi2:.1f}, p < 0,001 (p = {p:.2g}), V de Cramér = {w:.2f}")
registra("Qui-quadrado de aderência: período do dia", f"χ²={chi2:.1f}", p, f"V={w:.2f}", "P3")

**Interpretação.** Rejeita-se H0 (χ²(3) ≈ 3.091, p < 0,001). A noite concentra 42,1% dos casos contra os 25% esperados, e o efeito é moderado (V ≈ 0,23). Na prática, o policiamento preventivo tem mais retorno no período noturno, de 18h às 23h.

### Teste 2 · O sexo da vítima está associado ao local do crime? (Perguntas 2 e 4)

- **H0:** sexo da vítima e local do fato (dentro da residência × demais locais) são independentes.
- **H1:** há associação.
- **Teste:** qui-quadrado de independência (duas variáveis categóricas, tabela 2×2, contagens esperadas grandes). Tamanho de efeito: V de Cramér.

In [ ]:
tab = pd.crosstab(d["SEXO DA VITIMA"], d["RESID"].map({1: "Dentro da residência", 0: "Outros locais"}))
chi2, p, dof, esp = stats.chi2_contingency(tab)
v = cramers_v(tab)
display(tab)
display((tab.div(tab.sum(axis=1), axis=0) * 100).round(1))
print(f"χ²({dof}) = {chi2:.1f}, p = {p:.3g}, V de Cramér = {v:.2f}; menor contagem esperada = {esp.min():.0f}")
registra("Qui-quadrado de independência: sexo × dentro da residência", f"χ²={chi2:.1f}", p, f"V={v:.2f}", "P2/P4")

**Interpretação.** Rejeita-se H0 (χ²(1) ≈ 660, p < 0,001). 41% das vítimas mulheres morreram dentro de casa, contra 14% dos homens; efeito pequeno a moderado (V ≈ 0,19). Isso sustenta políticas específicas para o ambiente doméstico (canais de denúncia, medidas protetivas), distintas das de segurança em via pública.

### Teste 3 · A idade média difere entre homens e mulheres?

- **H0:** μ(homens) = μ(mulheres).
- **H1:** μ(homens) ≠ μ(mulheres).
- **Teste:** t de Welch (duas médias, sem supor variâncias iguais). Como a idade é assimétrica, o resultado é conferido com Mann-Whitney. Efeito: d de Cohen.

In [ ]:
h_ = d.loc[d["SEXO DA VITIMA"] == "Masculino", "IDADE_NUMERICA"].dropna()
m_ = d.loc[d["SEXO DA VITIMA"] == "Feminino", "IDADE_NUMERICA"].dropna()
lev = stats.levene(h_, m_)
t, p = stats.ttest_ind(h_, m_, equal_var=False)
u, p_u = stats.mannwhitneyu(h_, m_)
sp = np.sqrt(((len(h_) - 1) * h_.var() + (len(m_) - 1) * m_.var()) / (len(h_) + len(m_) - 2))
cohen = (h_.mean() - m_.mean()) / sp
print(f"Médias: homens {h_.mean():.1f} | mulheres {m_.mean():.1f}; medianas {h_.median():.0f} | {m_.median():.0f}")
print(f"Levene p = {lev.pvalue:.3g} (variâncias {'diferentes' if lev.pvalue < ALPHA else 'semelhantes'})")
print(f"Welch: t = {t:.2f}, p = {p:.3g}, d de Cohen = {cohen:.2f}")
print(f"Mann-Whitney (robustez): U = {u:.3g}, p = {p_u:.3g}")
registra("t de Welch: idade homens × mulheres", f"t={t:.2f}", p, f"d={cohen:.2f}", "perfil")

**Interpretação.** Rejeita-se H0 (p < 0,001, e Mann-Whitney concorda): as mulheres vítimas são em média mais velhas (31,6 contra 29,4 anos; medianas 30 e 26). Mas o efeito é pequeno (d ≈ −0,18): são dois anos de diferença, e o perfil etário de homens e mulheres é parecido. O resultado é estatisticamente claro, porém pouco relevante para desenhar ações por faixa de idade.

### Teste 4 · A idade média difere conforme o instrumento? (ANOVA)

- **H0:** a idade média é igual nos três instrumentos (arma de fogo, arma branca, espancamento).
- **H1:** pelo menos um instrumento tem idade média diferente.
- **Teste:** ANOVA de uma via (três grupos). Como a idade é assimétrica e as variâncias podem diferir, confere-se com Kruskal-Wallis. Efeito: η².

In [ ]:
grupos = {i: d.loc[d["INSTRUMENTO UTILIZADO"] == i, "IDADE_NUMERICA"].dropna() for i in ["PAF", "BRANCA", "ESPANCAMENTO"]}
f, p = stats.f_oneway(*grupos.values())
hk, p_k = stats.kruskal(*grupos.values())
todos = pd.concat(grupos.values())
eta2 = sum(len(g) * (g.mean() - todos.mean()) ** 2 for g in grupos.values()) / ((todos - todos.mean()) ** 2).sum()
display(pd.DataFrame({k: [len(g), g.mean(), g.median()] for k, g in grupos.items()}, index=["n", "média", "mediana"]).T.round(1))
print(f"Levene p = {stats.levene(*grupos.values()).pvalue:.3g}")
print(f"ANOVA: F = {f:.1f}, p = {p:.3g}, η² = {eta2:.3f}")
print(f"Kruskal-Wallis (robustez): H = {hk:.1f}, p = {p_k:.3g}")
registra("ANOVA: idade × instrumento", f"F={f:.1f}", p, f"η²={eta2:.3f}", "perfil")

**Interpretação.** Rejeita-se H0 (F ≈ 286, p < 0,001; Kruskal-Wallis concorda). Vítimas de arma de fogo são mais jovens (média 28,4) que as de arma branca (33,4) e de espancamento (34,8). Porém o η² ≈ 0,03 indica que o instrumento explica só cerca de 3% da variação da idade: a associação é real, mas fraca. Observação: o Levene indica variâncias diferentes entre os grupos, o que é mais uma razão para confiar no Kruskal-Wallis.

### Teste 5 · A proporção de mulheres entre as vítimas mudou? (Pergunta 2)

- **H0:** a proporção de vítimas mulheres em 2013–2015 é igual à de 2023–2025.
- **H1:** as proporções são diferentes.
- **Teste:** comparação de duas proporções (qui-quadrado 2×2 com correção de continuidade). Efeito: diferença em pontos percentuais e razão de chances.

In [ ]:
a = d[d["ANO"].between(2013, 2015)]
b = d[d["ANO"].between(2023, 2025)]
tab = np.array([[a["MULHER_I"].sum(), len(a) - a["MULHER_I"].sum()],
                [b["MULHER_I"].sum(), len(b) - b["MULHER_I"].sum()]])
chi2, p, _, _ = stats.chi2_contingency(tab)
p1, p2 = tab[0, 0] / tab[0].sum(), tab[1, 0] / tab[1].sum()
orr = (tab[1, 0] / tab[1, 1]) / (tab[0, 0] / tab[0, 1])
print(f"2013–15: {p1*100:.1f}% ({tab[0,0]} de {tab[0].sum()}) | 2023–25: {p2*100:.1f}% ({tab[1,0]} de {tab[1].sum()})")
print(f"Diferença = {(p2-p1)*100:.1f} p.p.; razão de chances = {orr:.2f}")
print(f"χ²(1) = {chi2:.2f}, p = {p:.3g}")
registra("Duas proporções: % mulheres 2013–15 × 2023–25", f"χ²={chi2:.2f}", p, f"Δ={(p2-p1)*100:.1f} p.p.", "P2")

**Interpretação.** Rejeita-se H0 (χ²(1) ≈ 9,1, p = 0,003). A parcela de mulheres subiu de 6,2% para 7,9% (+1,7 p.p.; razão de chances 1,30). A queda geral de homicídios não beneficiou as mulheres na mesma medida: elas passaram a ser uma fatia maior das vítimas. É o argumento estatístico para tratar a violência contra a mulher como frente própria. O efeito, porém, é modesto em pontos percentuais.

### Teste 6 · A capital perdeu ou ganhou peso relativo? (Perguntas 1 e 6)

- **H0:** a parcela de casos ocorridos em Maceió é a mesma em 2013–2015 e em 2023–2025.
- **H1:** a parcela mudou.
- **Teste:** qui-quadrado 2×2 (comparação de duas proporções).

In [ ]:
mac_a = (a["CIDADE DO FATO"] == "Maceió").sum(); mac_b = (b["CIDADE DO FATO"] == "Maceió").sum()
tab = np.array([[mac_a, len(a) - mac_a], [mac_b, len(b) - mac_b]])
chi2, p, _, _ = stats.chi2_contingency(tab)
p1, p2 = mac_a / len(a), mac_b / len(b)
print(f"Parcela de Maceió: {p1*100:.1f}% (2013–15) → {p2*100:.1f}% (2023–25); Δ = {(p2-p1)*100:.1f} p.p.")
print(f"χ²(1) = {chi2:.2f}, p = {p:.3g}")
registra("Duas proporções: parcela de Maceió 2013–15 × 2023–25", f"χ²={chi2:.2f}", p, f"Δ={(p2-p1)*100:.1f} p.p.", "P1/P6")

**Interpretação.** Rejeita-se H0, mas com evidência fraca (χ²(1) ≈ 4,6, p = 0,032, perto do limite de 0,05). A parcela de Maceió subiu de 33,5% para 35,8% (+2,3 p.p.): a capital caiu um pouco menos que o conjunto do interior. A diferença é pequena e o p-valor está perto de α, então a conclusão prática é cautelosa: não há sinal de deslocamento forte da violência, apenas leve aumento do peso relativo da capital. Isso é compatível com a heterogeneidade vista entre municípios na seção 3.

### Teste 7 · As vítimas estão ficando mais velhas ao longo dos anos? (correlação)

- **H0:** não há correlação monotônica entre ano do fato e idade da vítima (ρ = 0).
- **H1:** ρ ≠ 0.
- **Teste:** correlação de Spearman (idade é assimétrica e ano é discreto, então Pearson seria inadequado).

In [ ]:
x = d.dropna(subset=["IDADE_NUMERICA"])
rho, p = stats.spearmanr(x["ANO"], x["IDADE_NUMERICA"])
print(f"Spearman: ρ = {rho:.3f}, p = {p:.3g}, n = {len(x)}")
print(x.groupby("ANO")["IDADE_NUMERICA"].median().astype(int).to_dict())
registra("Correlação de Spearman: ano × idade", f"ρ={rho:.3f}", p, f"ρ={rho:.3f}", "perfil")

**Interpretação.** Rejeita-se H0 (ρ ≈ 0,08, p < 0,001). A correlação é positiva mas muito fraca: a significância vem do tamanho da amostra (n ≈ 18 mil). A mediana subiu de 25 para 28 anos entre 2013 e 2025, em linha com o que o notebook 01 mostrou (queda de jovens de 12–17 anos, alta de adultos de 25–34). A tendência é real, porém gradual.

### Resumo dos testes

In [ ]:
resumo_testes = pd.DataFrame(resultados)
resumo_testes["p-valor"] = resumo_testes["p-valor"].map(lambda v: f"{v:.2e}" if v < 0.001 else f"{v:.3f}")
display(resumo_testes)

# Síntese

Números de 2013 a 2025 (18.741 vítimas). A base não tem população, então não há taxa por 100 mil habitantes e as comparações entre municípios usam números absolutos.

1. **A queda foi homogênea entre as cidades?** Não. As vítimas caíram 57,7% entre 2013 e 2025, e todos os 24 municípios com pelo menos 150 vítimas tiveram queda. A intensidade variou muito: Pilar (−71,7%), Murici (−71,0%) e Coruripe (−69,6%) caíram mais de dois terços, enquanto Rio Largo (−17,6%) e Maragogi (−16,1%) caíram pouco. Maceió (−44,7%) e Arapiraca (−44,0%) ficaram abaixo da queda do estado.
2. **Os feminicídios acompanharam a queda?** Não. As mortes de mulheres caíram 36% (148 → 94), contra 58% do total, e a parcela de mulheres entre as vítimas subiu de 6,6% para 9,9%. A diferença de 6,2% para 7,9% entre os triênios 2013–15 e 2023–25 é estatisticamente significativa (p = 0,003). Os feminicídios classificados oscilam entre 18 e 43 por ano desde 2016, sem queda clara. Antes de 2015 a categoria não existia, então a série classificada só vale a partir daí. Entre os feminicídios, 65% ocorreram dentro da residência.
3. **Mais à noite?** Sim. A noite (18h–23h) concentra 42,1% dos casos, contra 25% esperados se o horário fosse indiferente (V de Cramér ≈ 0,23). Não há excesso de registros às 00:00 que indique horário preenchido por padrão.
4. **Público ou privado?** A maioria acontece fora de casa: 48,2% em via ou espaço público e 24,2% no entorno da residência, contra 15,6% dentro da residência. Para mulheres o quadro se inverte: 41,3% dentro de casa, contra 13,8% dos homens (V ≈ 0,19).
5. **Municípios de maior e menor incidência.** A violência é muito concentrada: Maceió tem 32,6% das vítimas, os 5 maiores municípios 48,2% e os 10 maiores 57,3%. Em Maceió, 10 de 51 bairros respondem por 62,6% dos casos.
6. **Concentração espacial ao longo do tempo.** Não há sinal forte de deslocamento. A parcela de Maceió passou de 33,5% para 35,8% (p = 0,032, evidência fraca). Interior e capital caíram em ritmo parecido, com a capital um pouco atrás.

**Limitações:** sem população não há taxas; escolaridade (77% ausente) e ocupação (70% ausente) não foram usadas; local é texto livre agrupado em categorias; a base é um registro de ocorrências, não uma amostra aleatória, e os testes devem ser lidos junto com o tamanho do efeito.

In [ ]:
import os
os.makedirs("../dados_tratados", exist_ok=True)

colunas = [
    "ID_CONTROLE", "DATA DO FATO", "ANO", "HORA", "PERIODO", "CIDADE DO FATO", "BAIRRO DO FATO",
    "SEXO DA VITIMA", "COR/RACA DA VITIMA", "IDADE_NUMERICA", "FAIXA_ETARIA",
    "INSTRUMENTO UTILIZADO", "SUBJETIVIDADE COMPLEMENTAR", "LOCAL_GRUPO",
]
tratado = d[colunas]
tratado.to_csv("../dados_tratados/cvli_tratado_2013_2025.csv", index=False, encoding="utf-8-sig")
print(tratado.shape)